# I. DATA INGESTION & CLEANING

## •	Loading the dataset with Pandas

In [0]:
import pandas as pd
import numpy as np

In [0]:
df = spark.read.table("workspace.default.nyc").toPandas()

In [0]:
df.head(5)

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id1589506,2,2016-04-02 17:41:37,2016-04-02 17:57:33,5,-73.984810,40.779400,-73.959007,40.819759,N,956
1,id1081378,1,2016-05-17 03:41:22,2016-05-17 03:52:33,1,-74.002853,40.724976,-73.984329,40.691662,N,671
2,id2988321,2,2016-04-01 08:19:39,2016-04-01 08:27:36,1,-73.978630,40.744942,-73.988564,40.737259,N,477
3,id2759902,1,2016-06-27 06:00:56,2016-06-27 06:43:14,1,-73.878700,40.737270,-73.938126,40.828178,N,2538
4,id0965960,1,2016-03-22 08:02:22,2016-03-22 08:48:16,1,-73.871078,40.773678,-73.982933,40.735069,N,2754


In [0]:
df.info(memory_usage = "deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458644 entries, 0 to 1458643
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   id                  1458644 non-null  object        
 1   vendor_id           1458644 non-null  int64         
 2   pickup_datetime     1458644 non-null  datetime64[ns]
 3   dropoff_datetime    1458644 non-null  datetime64[ns]
 4   passenger_count     1458644 non-null  int64         
 5   pickup_longitude    1458644 non-null  float64       
 6   pickup_latitude     1458644 non-null  float64       
 7   dropoff_longitude   1458644 non-null  float64       
 8   dropoff_latitude    1458644 non-null  float64       
 9   store_and_fwd_flag  1458644 non-null  object        
 10  trip_duration       1458644 non-null  int64         
dtypes: datetime64[ns](2), float64(4), int64(3), object(2)
memory usage: 250.4 MB


## •	Handle missing values and invalid coordinates

In [0]:
df.isna().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

_**No missing values**_
<hr>

_**To validate the longtitide and latitude values we need to check the column values. Latitude need to be between -90 to 90 and Longtitude from -180 to 180.**_

In [0]:
df_filtered_columns = df[["pickup_longitude","pickup_latitude","dropoff_longitude","dropoff_latitude"]]
df_filtered = df_filtered_columns.query("pickup_latitude >= -90 and pickup_latitude <= 90 &"
                                        "dropoff_latitude >= -90 and dropoff_latitude <= 90 &"
                                        "pickup_longitude >= -180 and pickup_longitude <= 180 &"
                                        "dropoff_longitude >= -180 and dropoff_longitude <= 180")
df_filtered_columns.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458644 entries, 0 to 1458643
Data columns (total 4 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   pickup_longitude   1458644 non-null  float64
 1   pickup_latitude    1458644 non-null  float64
 2   dropoff_longitude  1458644 non-null  float64
 3   dropoff_latitude   1458644 non-null  float64
dtypes: float64(4)
memory usage: 44.5 MB


_**We can see from the df_filtered_columns.info() result that the longtitude and latitude filtered column rows are the exact same number as the initial data set(1458644 entries).Also, null values are checked above with isna.sum(). Than means all the values are valid.**_
<hr>

## • Filter or remove extreme outliers

_**For identifing outliers, first we need to be searching the appropriate columns and setting the approriate metrics.**_

_**1. Passengers: Passengers need to be more than 0 and less than or equal to 6 <br>
2. Trip Duration: Trips below 60 seconds are not entirely valid and we set an upper limit of 12 hours (43,200 seconds). <br>
3. Datetimes: Drop off times can't be less pickup times for obvious reasons.**_

In [0]:
df.describe().round(2)

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration
count,1458644.00,1458644,1458644,1458644.00,1458644.00,1458644.00,1458644.00,1458644.00,1458644.00
mean,1.53,2016-04-01 10:10:24.940037120,2016-04-01 10:26:24.432310784,1.66,-73.97,40.75,-73.97,40.75,959.49
min,1.00,2016-01-01 00:00:17,2016-01-01 00:03:31,0.00,-121.93,34.36,-121.93,32.18,1.00
25%,1.00,2016-02-17 16:46:04.249999872,2016-02-17 17:05:32.500000,1.00,-73.99,40.74,-73.99,40.74,397.00
50%,2.00,2016-04-01 17:19:40,2016-04-01 17:35:12,1.00,-73.98,40.75,-73.98,40.75,662.00
75%,2.00,2016-05-15 03:56:08.750000128,2016-05-15 04:10:51.750000128,2.00,-73.97,40.77,-73.96,40.77,1075.00
max,2.00,2016-06-30 23:59:39,2016-07-01 23:02:03,9.00,-61.34,51.88,-61.34,43.92,3526282.00
std,0.50,NaN,NaN,1.31,0.07,0.03,0.07,0.04,5237.43


**_1. Passengers: Passengers need to be more than 0 and less than or equal to 6_**

In [0]:
passenger_zero = df[["passenger_count"]].query("passenger_count == 0 or passenger_count > 6")
passenger_zero.count()


passenger_count    65
dtype: int64

_**We currently have 65 rows with ouliers. 60 equal to 0, 5 greater than 6**_ <hr>

**_2. Trip Duration: Trips below to 60 seconds are not entirely valid and we set an upper limit of 12 hours (43,200 seconds)._**

In [0]:
invalid_trips = df[['trip_duration']].query("trip_duration == 0 or trip_duration <= 60 or trip_duration > 43200")
invalid_trips.count()

trip_duration    10770
dtype: int64

**_We have 10770 trips below 60 and above 43,200 seconds._** <hr>

_**3. Datetimes: Drop off times can't be less pickup times for obvious reasons.**_

In [0]:
invalid_times = df[['pickup_datetime','dropoff_datetime']].query("pickup_datetime > dropoff_datetime")
invalid_times.count()

pickup_datetime     0
dropoff_datetime    0
dtype: int64

_**No anomalies in those columns**_ <hr>

_**Finally we combine the querries to clean the data in a new data frame**_

In [0]:

df_clean = df.query("passenger_count >= 1 and passenger_count <= 6 and "
                     "trip_duration > 60 and trip_duration <= 43200 and "
                     "pickup_datetime <= dropoff_datetime")


print(f"Original dataset: {len(df):,} rows")
print(f"Cleaned dataset: {len(df_clean):,} rows")
print(f"Removed: {len(df) - len(df_clean):,} rows ({(len(df) - len(df_clean))/len(df)*100:.2f}%)")

Original dataset: 1,458,644 rows
Cleaned dataset: 1,447,855 rows
Removed: 10,789 rows (0.74%)


## •	Feature engineering: <br>
##Extract pickup_hour, pickup_day, pickup_month from pickup_datetime


In [0]:
df_clean = df_clean.assign( pickup_hour = df_clean["pickup_datetime"].dt.hour,
                            pickup_day = df_clean["pickup_datetime"].dt.day_name().str.slice(stop=3),
                            pickup_month = df_clean["pickup_datetime"].dt.month_name().str.slice(stop=3)
)
                 
df_clean.head(5)

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_hour,pickup_day,pickup_month
0,id1589506,2,2016-04-02 17:41:37,2016-04-02 17:57:33,5,-73.984810,40.779400,-73.959007,40.819759,N,956,17,Sat,Apr
1,id1081378,1,2016-05-17 03:41:22,2016-05-17 03:52:33,1,-74.002853,40.724976,-73.984329,40.691662,N,671,3,Tue,May
2,id2988321,2,2016-04-01 08:19:39,2016-04-01 08:27:36,1,-73.978630,40.744942,-73.988564,40.737259,N,477,8,Fri,Apr
3,id2759902,1,2016-06-27 06:00:56,2016-06-27 06:43:14,1,-73.878700,40.737270,-73.938126,40.828178,N,2538,6,Mon,Jun
4,id0965960,1,2016-03-22 08:02:22,2016-03-22 08:48:16,1,-73.871078,40.773678,-73.982933,40.735069,N,2754,8,Tue,Mar


- ## EXTRA! <br>
## Memory optimization

_**First we check which data we can downcast**_

In [0]:
df_clean.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 1447855 entries, 0 to 1458643
Data columns (total 14 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   id                  1447855 non-null  object        
 1   vendor_id           1447855 non-null  int64         
 2   pickup_datetime     1447855 non-null  datetime64[ns]
 3   dropoff_datetime    1447855 non-null  datetime64[ns]
 4   passenger_count     1447855 non-null  int64         
 5   pickup_longitude    1447855 non-null  float64       
 6   pickup_latitude     1447855 non-null  float64       
 7   dropoff_longitude   1447855 non-null  float64       
 8   dropoff_latitude    1447855 non-null  float64       
 9   store_and_fwd_flag  1447855 non-null  object        
 10  trip_duration       1447855 non-null  int64         
 11  pickup_hour         1447855 non-null  int32         
 12  pickup_day          1447855 non-null  object        
 13  pickup_month     

_**We can see that the initial 250.4MB of memory have turned to 408.7MB due to the addition of the pickup_datetime data.**_

_**'We can safely downcast all integers according to theie max values, and objects to Category/String**_

* id --> _'object'_
* vendor_id --> _'int64'_
* Passenger_count --> _'int64'_
* store_and_fwd_flag --> _'object'_
* trip_duration --> _'int64'_
* pickup_hour  --> _'int32'_        
* pickup_day  -->  _'object'_    
* pickup_month  --> _'object'_

In [0]:
df_clean.describe().loc[["max"]].T.round(2)

,max
vendor_id,2.0
pickup_datetime,2016-06-30 23:59:39
dropoff_datetime,2016-07-01 00:48:20
passenger_count,6.0
pickup_longitude,-61.335529
pickup_latitude,51.881084
dropoff_longitude,-61.335529
dropoff_latitude,43.921028
trip_duration,43177.0
pickup_hour,23.0


**_• Vendor_id, passenger_count and pickup_hour are well within the range of 8bits (-128 to 127), with a lot of space for data base expansion_ <br>**
**_• Trip_duration went from 3,526,282 to 43,177 as max value, since we trimmed ouliers but still is cast as int64. Int16 ranges from (-32,768 to 32,767), so we will use int32 instead._**

In [0]:
df_clean= df_clean.astype({"vendor_id":"int8",
                "passenger_count":"int8",
                "trip_duration":"int32",
                "pickup_hour":"int8"})
 

In [0]:
df_clean.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 1447855 entries, 0 to 1458643
Data columns (total 14 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   id                  1447855 non-null  object        
 1   vendor_id           1447855 non-null  int8          
 2   pickup_datetime     1447855 non-null  datetime64[ns]
 3   dropoff_datetime    1447855 non-null  datetime64[ns]
 4   passenger_count     1447855 non-null  int8          
 5   pickup_longitude    1447855 non-null  float64       
 6   pickup_latitude     1447855 non-null  float64       
 7   dropoff_longitude   1447855 non-null  float64       
 8   dropoff_latitude    1447855 non-null  float64       
 9   store_and_fwd_flag  1447855 non-null  object        
 10  trip_duration       1447855 non-null  int32         
 11  pickup_hour         1447855 non-null  int8          
 12  pickup_day          1447855 non-null  object        
 13  pickup_month     

In [0]:
df_clean[["id"]].nunique()/len(df)*100

id    99.26034
dtype: float64

_**The 'id' column is 99.2% unique so we will use it a string (pyarrow) type to save some memory space**_

In [0]:
df_clean[["store_and_fwd_flag"]].nunique()/len(df)*100

store_and_fwd_flag    0.000137
dtype: float64

In [0]:
df_clean[["store_and_fwd_flag"]].nunique()

store_and_fwd_flag    2
dtype: int64

**_'store_and_fwd_flag' has only 2 unique values in all rows so we can cast it as a category type._**

In [0]:
df_clean[["pickup_day"]].nunique()/len(df)*100

pickup_day    0.00048
dtype: float64

In [0]:
df_clean[["pickup_month"]].nunique()/len(df)*100

pickup_month    0.000411
dtype: float64

**_We know for a fact that pickup_day and pickup_month columns, have 6 and 12 unique values respectively, out of the 1447855 total entries in the dataset. Those will be cast as categories too._**

In [0]:
df_clean= df_clean.astype({"id":"string[pyarrow]",
                          "store_and_fwd_flag":"category",
                          "pickup_day": "object",
                          "pickup_month": "object"})

In [0]:
df_clean.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 1447855 entries, 0 to 1458643
Data columns (total 14 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   id                  1447855 non-null  string        
 1   vendor_id           1447855 non-null  int8          
 2   pickup_datetime     1447855 non-null  datetime64[ns]
 3   dropoff_datetime    1447855 non-null  datetime64[ns]
 4   passenger_count     1447855 non-null  int8          
 5   pickup_longitude    1447855 non-null  float64       
 6   pickup_latitude     1447855 non-null  float64       
 7   dropoff_longitude   1447855 non-null  float64       
 8   dropoff_latitude    1447855 non-null  float64       
 9   store_and_fwd_flag  1447855 non-null  category      
 10  trip_duration       1447855 non-null  int32         
 11  pickup_hour         1447855 non-null  int8          
 12  pickup_day          1447855 non-null  object        
 13  pickup_month     

_**By downcasting the data types we went from 408.7MB of memory to 255.4MB**_

#II. DATA VISUALIZATION & EXPLORATORY ANALYSIS

## ⚠️ The graphs were built in Light Theme. Dark themes will invert the colors
<hr>


In [0]:
import plotly.express as px
import plotly.graph_objects as go

## Trip Distribution Analysis

- ###  Trips per hour

In [0]:
df_clean

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_hour,pickup_day,pickup_month
0,id1589506,2,2016-04-02 17:41:37,2016-04-02 17:57:33,5,-73.984810,40.779400,-73.959007,40.819759,N,956,17,Sat,Apr
1,id1081378,1,2016-05-17 03:41:22,2016-05-17 03:52:33,1,-74.002853,40.724976,-73.984329,40.691662,N,671,3,Tue,May
2,id2988321,2,2016-04-01 08:19:39,2016-04-01 08:27:36,1,-73.978630,40.744942,-73.988564,40.737259,N,477,8,Fri,Apr
3,id2759902,1,2016-06-27 06:00:56,2016-06-27 06:43:14,1,-73.878700,40.737270,-73.938126,40.828178,N,2538,6,Mon,Jun
4,id0965960,1,2016-03-22 08:02:22,2016-03-22 08:48:16,1,-73.871078,40.773678,-73.982933,40.735069,N,2754,8,Tue,Mar
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1458639,id2376096,2,2016-04-08 13:31:04,2016-04-08 13:44:02,4,-73.982201,40.745522,-73.994911,40.740170,N,778,13,Fri,Apr
1458640,id1049543,1,2016-01-10 07:35:15,2016-01-10 07:46:10,1,-74.000946,40.747379,-73.970184,40.796547,N,655,7,Sun,Jan
1458641,id2304944,2,2016-04-22 06:57:41,2016-04-22 07:10:25,1,-73.959129,40.768799,-74.004433,40.707371,N,764,6,Fri,Apr
1458642,id2714485,1,2016-01-05 15:56:26,2016-01-05 16:02:39,1,-73.982079,40.749062,-73.974632,40.757107,N,373,15,Tue,Jan


In [0]:
trips_per_hour = df_clean.groupby(["pickup_hour"])[["id"]].count().reset_index(names = "hours")
trips_per_hour

,hours,id
0,0,52764
1,1,38215
2,2,27651
3,3,20636
4,4,15522
5,5,14767
6,6,32936
7,7,55217
8,8,66659
9,9,67246


In [0]:
fig = px.bar(trips_per_hour,
              x= "hours",
              y="id",
              labels=dict(hours="<b>Hours</b>", id="<b>Trips Count</b>"),
              text="id",
              title="<b><i>Tripes per Hour</b></i>",
             
              color_discrete_sequence=["#ffae1b"])

fig.update_traces(base=dict(width = 3),
                    marker_line_color="#b300b3",
                    marker_line_width=1.75,
                    textfont_size=13)
        
               
    
fig.update_layout(plot_bgcolor="#fafafa",
                  xaxis_title_font=dict(size=16),
                  yaxis_title_font=dict(size=16),
                  xaxis=dict(tickfont=dict(size=16)),
                  yaxis=dict(tickfont=dict(size=16)))
                  #width=1000,
                  #height=600)

fig.update_layout(xaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
                  yaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
                  plot_bgcolor="#f9f9f9")
fig.show()


- ### Trips per day

In [0]:
trips_per_day = df_clean.groupby(["pickup_day"])[["id"]].count().reset_index(names = "day")

# We need to create a custom category to sort the day of the week, else they are going to be sorted aplheticaly, since we have cast the pickup_day as category type

day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]   
trips_per_day["day"] = pd.Categorical(trips_per_day["day"], categories=day_order, ordered=True) 
trips_per_day = trips_per_day.sort_values("day") 
trips_per_day

,day,id
1,Mon,186064
5,Tue,201333
6,Wed,208763
4,Thu,216951
0,Fri,221867
2,Sat,219190
3,Sun,193687


In [0]:
fig = px.bar(trips_per_day,
              x= "day",
              y="id",
              labels=dict(day="<b>Day</b>", id="<b>Trips Count</b>"),
              text="id",
              title="<b><i>Daily Trips</b></i>",
             
              color_discrete_sequence=["#ffae1b"])

fig.update_traces(base=dict(width = 3),
                    marker_line_color="#b300b3",
                    marker_line_width=1.75,
                    textfont_size=13)
        
               
    
fig.update_layout(plot_bgcolor="#fafafa",
                  xaxis_title_font=dict(size=16),
                  yaxis_title_font=dict(size=16),
                  xaxis=dict(tickfont=dict(size=16)),
                  yaxis=dict(tickfont=dict(size=16)))


fig.update_layout(xaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
                  yaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
                  plot_bgcolor="#f9f9f9")
fig.show()


- ###  Passenger count distribution

In [0]:
passenger_dist = df_clean.groupby(["passenger_count"])[["id"]].count().reset_index(names = "passenger_count")
passenger_dist



,passenger_count,id
0,1,1025347
1,2,209115
2,3,59548
3,4,28226
4,5,77585
5,6,48034


In [0]:
fig = px.bar(passenger_dist,
              x= "passenger_count",
              y="id",
              labels=dict(passenger_count="<b>Passenger Count</b>", id="<b>Trips Count</b>"),
              text="id",
              title="<b><i>Passenger Distribution</b></i>",
             
              color_discrete_sequence=["#ffae1b"])


fig.update_traces(marker_line_color="#b300b3",
                    marker_line_width=1.75,
                    textfont_size=13,
                    textposition='auto')
        
               
    
fig.update_layout(plot_bgcolor="#fafafa",
                  xaxis_title_font=dict(size=16),
                  yaxis_title_font=dict(size=16),
                  xaxis=dict(tickfont=dict(size=16)),
                  yaxis=dict(tickfont=dict(size=16)))


fig.update_layout(xaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
                  yaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
                  plot_bgcolor="#f9f9f9")
fig.show()

## Trip Duration vs Passenger Count

- ### Scatter plot with optional regression line

In [0]:
x = df_clean["passenger_count"]
y = df_clean["trip_duration"]
slope, intercept = np.polyfit(x, y, 1)
print(f"Regression equation: Trip duration = {intercept:.2f} + {slope:.2f} * Passenger count")

Regression equation: Trip duration = 831.77 + 7.11 * Passenger count


In [0]:
df_sample = df_clean.sample(n=10000, random_state=42)
x_sample = df_sample["passenger_count"]
y_sample = df_sample["trip_duration"]
slope, intercept = np.polyfit(x_sample, y_sample, 1)
print(f"Regression equation: Trip duration = {intercept:.2f} + {slope:.2f} * Passenger count")

Regression equation: Trip duration = 842.47 + 4.59 * Passenger count


In [0]:
%pip install statsmodels

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Manual 'jitter' to spread the values 
df_clean["passenger_jitter"] = df_clean["passenger_count"] + np.random.uniform(-0.5, 0.5, len(df_clean))

fig = px.scatter(df_clean, 
                 x="passenger_jitter", 
                 y="trip_duration",
                 color="passenger_count",
                 color_continuous_scale="Rainbow",
                 trendline="ols",
                 trendline_color_override="black", 
                 labels=dict(passenger_jitter="<b>Passenger Count</b>", 
                             trip_duration="<b>Duration (s)</b>"),
                 title="<b><i>Passenger Count vs Trip Duration</b></i> (Jittered)",
                 opacity=0.5)

fig.update_traces(line=dict(width=3), selector=dict(mode='lines'))

# Regression line
fig.add_annotation(
    x=df_sample["passenger_count"].median(),
    y=df_sample["trip_duration"].mean(),
    text="<b>Regression Line</b>",
    showarrow=False,
    yshift=15,             # Pulls text slightly above the line
    font=dict(color="black", size=18))

# Force the axis to normal count
fig.update_layout(
    xaxis = dict(
        tickmode = "array",
        tickvals = [1, 2, 3, 4, 5, 6],
        ticktext = ["1", "2", "3", "4", "5", "6"],
        showgrid=True, 
        gridcolor="#e6e6e6"),
    yaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
    plot_bgcolor="#f9f9f9")

fig.show()

*** WARNING: Output too large, the following mime types were removed from the output: text/html. ***

## ⚠️ Due to the volume of values html isn't able to put all the entries in graphics. instead, we use a sample of 10,000 values 
<hr>


In [0]:
%pip install statsmodels

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Manual 'jitter' to spread the values 
df_sample["passenger_jitter"] = df_sample["passenger_count"] + np.random.uniform(-0.5, 0.5, len(df_sample))

fig = px.scatter(df_sample, 
                 x="passenger_jitter", 
                 y="trip_duration",
                 color="passenger_count",
                 color_continuous_scale="Rainbow",
                 trendline="ols",
                 trendline_color_override="black", 
                 labels=dict(passenger_jitter="<b>Passenger Count</b>", 
                             trip_duration="<b>Duration (s)</b>",
                             passenger_count="<b>Passenger Count</b>"),
                 title="<b><i>Passenger Count vs Trip Duration</b></i> (Jittered)",
                 opacity=0.5)

fig.update_traces(line=dict(width=3), selector=dict(mode='lines'))
fig.update_coloraxes(showscale=False)

# Regression line
fig.add_annotation(
    x=df_sample["passenger_count"].median(),
    y=df_sample["trip_duration"].mean(),
    text="<b>Regression Line</b>",
    showarrow=False,
    yshift=15,             # Pulls text slightly above the line
    font=dict(color="black", size=18))

# Force the axis to normal count
fig.update_layout(
    xaxis = dict(
        tickmode = "array",
        tickvals = [1, 2, 3, 4, 5, 6],
        ticktext = ["1", "2", "3", "4", "5", "6"],
        showgrid=True, 
        gridcolor="#e6e6e6"),
    yaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
    plot_bgcolor="#f9f9f9")

fig.show()

_**We use 'jitter' to spread the values along the x axis, otherwise all y values would get 'stacked' on a straight line along the x axis values**_

- ### Explain qualitative correlation between trip duration and passenger count

In [0]:
correlation = df_clean["passenger_count"].corr(df_clean["trip_duration"])
print(f"Correlation coefficient (r): {correlation:.4f}")

Correlation coefficient (r): 0.0134


**_Here, we use the original data frame for the correlation although numbers are really close with the df_sample coefficient being at 0.0092_**

### A Correlation coefficient of 0.0134, shows that passenger count and trip duration have
###  a close to 0 relation. That means that our variables are reliable not to eachother. <hr>

- ## Geospatial Analysis

- ### Heatmap of pickup locations

In [0]:

fig = go.Figure(go.Densitymap(lat=df_sample["pickup_latitude"], 
                              lon=df_sample["pickup_longitude"],
                              radius=5,
                              colorscale="Viridis",
                              showscale=False,
                # Extra data on hover                                
                              customdata=df_sample[["passenger_count", "trip_duration"]], 
                              hovertemplate=(
                                    "<b>Location Info</b><br>" +
                                    "Latitude: %{lat:.4f}<br>" +
                                    "Longitude: %{lon:.4f}<br>" +
                                    "Passengers: %{customdata[0]}<br>" +
                                    "Duration: %{customdata[1]}s" +
                                    "<extra></extra>"))) #  Remove the trace box

fig.update_layout(map_style="open-street-map", map_zoom=10,
    map_center=dict(
        lat=df_sample["pickup_latitude"].mean(), 
        lon=df_sample["pickup_longitude"].mean()))
    
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

fig.show()

- ### Heatmap of dropoff locations

In [0]:
fig = go.Figure(go.Densitymap(lat=df_sample["dropoff_latitude"],
                              lon=df_sample["dropoff_longitude"],
                              radius=5,
                              colorscale="Viridis",
                              showscale=False,
                    # Extra data on hover                                
                                customdata=df_sample[["passenger_count", "trip_duration"]], 
                                hovertemplate=(
                                    "<b>Location Info</b><br>" +
                                    "Latitude: %{lat:.4f}<br>" +
                                    "Longitude: %{lon:.4f}<br>" +
                                    "Passengers: %{customdata[0]}<br>" +
                                    "Duration: %{customdata[1]}s" +
                                    "<extra></extra>"))) #  Remove the trace box

fig.update_layout(map_style="open-street-map", map_zoom=10,
    map_center=dict(
        lat=df_sample["dropoff_latitude"].mean(), 
        lon=df_sample["dropoff_longitude"].mean()))
    
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

- ## Traffic Pattern Analysis

- ### Compare peak hours vs off-peak hours

In [0]:

peak_hours = (df_clean
              .groupby("pickup_hour")
              .agg(id_count=("id", "count"))
              .reset_index())


peak_hours["peak_status"] = np.where(
    peak_hours['id_count'] > peak_hours['id_count'].mean(), "Peak", "Off-Peak")


peak_hours = (peak_hours
                .set_index(["pickup_hour", "peak_status"])
                .sort_values(by="id_count", ascending=False)
                .reset_index())
peak_hours

,pickup_hour,peak_status,id_count
0,18,Peak,90025
1,19,Peak,89747
2,21,Peak,83608
3,20,Peak,83545
4,22,Peak,79931
5,17,Peak,75928
6,14,Peak,73722
7,12,Peak,71371
8,15,Peak,71242
9,13,Peak,70988


In [0]:
fig = px.scatter(peak_hours,
               x="id_count",
               y="pickup_hour",
               color="peak_status",
               labels=dict(pickup_hour= "<b>Hour</b>", id_count="<b>Trips</b>"),
               title="<b><i>Peak Hours</b></i>",
               color_discrete_map={
                   "Peak": "#b300b3" , 
                   "Off-Peak": "#ffae1b"
               })


fig.update_traces(marker=dict(symbol="diamond",size=8))
              

fig.update_layout(plot_bgcolor="#fafafa",
                   xaxis_title_font=dict(size=16),
                   yaxis_title_font=dict(size=16),
                   xaxis=dict(tickmode="linear", dtick=10000, tickfont=dict(size=16)),
                   yaxis=dict(dtick=4, tickfont=dict(size=16)),
                   legend_title_text="Peak Status")
                  

fig.update_layout(xaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
                  yaxis=dict(showgrid=True, gridcolor="#e6e6e6"),
                  plot_bgcolor="#f9f9f9")

fig.show()

- ### Compare weekday vs weekend traffic

In [0]:

# First we need to create the 'weekday' 'weekend' column
df_clean["weekend"] = np.where(df_clean['pickup_day'].isin(['Sat', 'Sun']), "Weekend", "Weekday")

# We create the custom category as above
day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]   
df_clean["pickup_day"] = pd.Categorical(df_clean["pickup_day"], categories=day_order, ordered=True)


weekdays_weekends = (df_clean
                     .groupby(["weekend", "pickup_day"], observed=True)
                     .agg(trip_count=("id", "count"))
                     .sort_values(by=["weekend"]))

weekdays_weekends

trip_count
weekend pickup_day            
Weekday Mon             186064
        Tue             201333
        Wed             208763
        Thu             216951
        Fri             221867
Weekend Sat             219190
        Sun             193687

In [0]:
plot_data = weekdays_weekends.reset_index()

fig = px.sunburst(plot_data, 
                  path=["weekend", "pickup_day"], 
                  values="trip_count",
                  title="<b>Trip Distribution: Weekday vs. Weekend</b>",
                  color="weekend", 
                  color_discrete_map={
                      "Weekday": "#ffae1b",  
                      "Weekend": "#b300b3"})

fig.update_traces(
    textinfo="label+value",
    texttemplate="<b>%{label}</b><br>%{value:,} trips")

fig.update_layout(width=700, height=700)
fig.show()

/databricks/python/lib/python3.12/site-packages/plotly/express/_core.py:1727: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



# III. DESCRIPTIVE STATISTICS & ANALYTICS

- ### 	What is the average trip duration per hour?

In [0]:
df_clean

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_hour,pickup_day,pickup_month,passenger_jitter,weekend
0,id1589506,2,2016-04-02 17:41:37,2016-04-02 17:57:33,5,-73.984810,40.779400,-73.959007,40.819759,N,956,17,Sat,Apr,4.805727,Weekend
1,id1081378,1,2016-05-17 03:41:22,2016-05-17 03:52:33,1,-74.002853,40.724976,-73.984329,40.691662,N,671,3,Tue,May,0.748612,Weekday
2,id2988321,2,2016-04-01 08:19:39,2016-04-01 08:27:36,1,-73.978630,40.744942,-73.988564,40.737259,N,477,8,Fri,Apr,0.546535,Weekday
3,id2759902,1,2016-06-27 06:00:56,2016-06-27 06:43:14,1,-73.878700,40.737270,-73.938126,40.828178,N,2538,6,Mon,Jun,0.599851,Weekday
4,id0965960,1,2016-03-22 08:02:22,2016-03-22 08:48:16,1,-73.871078,40.773678,-73.982933,40.735069,N,2754,8,Tue,Mar,1.116824,Weekday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1458639,id2376096,2,2016-04-08 13:31:04,2016-04-08 13:44:02,4,-73.982201,40.745522,-73.994911,40.740170,N,778,13,Fri,Apr,4.037622,Weekday
1458640,id1049543,1,2016-01-10 07:35:15,2016-01-10 07:46:10,1,-74.000946,40.747379,-73.970184,40.796547,N,655,7,Sun,Jan,0.766413,Weekend
1458641,id2304944,2,2016-04-22 06:57:41,2016-04-22 07:10:25,1,-73.959129,40.768799,-74.004433,40.707371,N,764,6,Fri,Apr,1.314151,Weekday
1458642,id2714485,1,2016-01-05 15:56:26,2016-01-05 16:02:39,1,-73.982079,40.749062,-73.974632,40.757107,N,373,15,Tue,Jan,0.946073,Weekday


In [0]:
trip_duration_ph = df_clean.groupby(["pickup_hour"])[["trip_duration"]].mean()
trip_duration_ph

,trip_duration
pickup_hour,
0,786.099879
1,745.232788
2,707.497342
3,709.043032
4,744.506442
5,722.139568
6,677.240557
7,763.257185
8,838.547998


- ### What is the average trip duration per day of the week?

In [0]:
avg_trip_duration_pw = df_clean.groupby(["pickup_day"])[["trip_duration"]].mean()
avg_trip_duration_pw

/home/spark-c212f272-a52b-4e5a-a452-ef/.ipykernel/2936/command-7112878563927583-1212453129:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,trip_duration
pickup_day,
Mon,817.053852
Tue,862.552239
Wed,884.903355
Thu,904.019299
Fri,873.628615
Sat,784.364177
Sun,769.948200


- ### Which pickup location has the most trips?

In [0]:
most_trips_location = (df_clean
                       .groupby(["pickup_longitude", "pickup_latitude"])[["id"]].count()
                       .sort_values(by=["id"], ascending=False)
                       .head())
most_trips_location

,,id
pickup_longitude,pickup_latitude,
-73.954666,40.821003,39
-73.870934,40.773788,15
-73.873009,40.774181,14
-73.870934,40.773769,14
-73.870872,40.773739,14


- ### Which trips are the longest and shortest?

In [0]:
longest_trips = (df_clean
                       .groupby(["pickup_longitude", "pickup_latitude"])[["trip_duration"]].max()
                       .sort_values(by=["trip_duration"], ascending=False)
                       .head())
longest_trips


,,trip_duration
pickup_longitude,pickup_latitude,
-73.934792,40.593227,43177
-73.978333,40.752384,42098
-73.789383,40.643120,41356
-73.972191,40.754681,40399
-73.937531,40.813000,40357


In [0]:
shortest_trips = (df_clean
                       .groupby(["pickup_longitude", "pickup_latitude"])[["trip_duration"]].min()
                       .sort_values(by=["trip_duration"], ascending=True)
                       .head(10))
shortest_trips


,,trip_duration
pickup_longitude,pickup_latitude,
-73.981613,40.778316,61
-74.008087,40.737953,61
-73.995621,40.759762,61
-73.959679,40.762711,61
-73.986511,40.734100,61
-73.976562,40.780685,61
-73.981583,40.741093,61
-73.986038,40.767765,61
-73.988113,40.718761,61


- ### Which hour of day has the highest traffic volume?

In [0]:
peak_hours.head()

,pickup_hour,peak_status,id_count
0,18,Peak,90025
1,19,Peak,89747
2,21,Peak,83608
3,20,Peak,83545
4,22,Peak,79931


- ### What is the correlation coefficient between trip duration and passenger count?

In [0]:
correlation = df_clean["passenger_count"].corr(df_clean["trip_duration"])
print(f"Correlation coefficient (r): {correlation:.4f}")

Correlation coefficient (r): 0.0134


### A Correlation coefficient of 0.0134, shows that passenger count and trip duration have
###  a close to 0 relation. That means that our variables are reliable not to eachother. <hr>

# IV. INTERACTIVE PYTHON DASHBOARD

### We are going to be working on the dashboard though Data Bricks built-in app. 
### we just need to save the data frames we are going to use.  <hr>

In [0]:
# Save df_clean and df_sample to Unity Catalog 
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


df_clean_spark = spark.createDataFrame(df_clean)
df_clean_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.nyc_clean")
print(f"✓ Saved df_clean ({len(df_clean):,} rows)")

df_sample_spark = spark.createDataFrame(df_sample)
df_sample_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.nyc_sample")
print(f"✓ Saved df_sample ({len(df_sample):,} rows)")

print("\n✓ Tables saved successfully to workspace.default")

✓ Saved df_clean (1,447,855 rows)
✓ Saved df_sample (10,000 rows)

✓ Tables saved successfully to workspace.default


In [0]:
# Save to Unity Catalog for dashboard use
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()


df_most_trips = spark.createDataFrame(most_trips_location.reset_index())
df_most_trips.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.default.nyc_most_trips_location")
print(f" Saved most_trips_location")


df_longest = spark.createDataFrame(longest_trips.reset_index())
df_longest.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.default.nyc_longest_trips")
print(f"Saved longest_trips")


df_shortest = spark.createDataFrame(shortest_trips.reset_index())
df_shortest.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.default.nyc_shortest_trips")
print(f" Saved shortest_trips")

print("\n All dataframes created and saved to Unity Catalog!")

 Saved most_trips_location
Saved longest_trips
 Saved shortest_trips

 All dataframes created and saved to Unity Catalog!
